# Entropy & Alignment Statistics

Aggregated tables supporting the information-theoretic analysis (entropy figure).
Each table is displayed in pandas and exported as LaTeX.

In [1]:
import sys, numpy as np, pandas as pd
from pathlib import Path
from scipy import stats
from scipy.stats import entropy as sp_entropy
from IPython.display import display, Latex

ROOT = Path('.').resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'analysis'))
from config import MODELS_7B
from utils.constants import VARIANT_ORDER

EXPORTS = ROOT / 'analysis/session2/exports'
OUT_DIR = Path('.').resolve()
_7b = set(MODELS_7B)

pc = pd.read_parquet(EXPORTS / 'pair_cache.parquet')
human = pd.read_csv(EXPORTS / 'responses_human.csv')
model_ib = pd.read_csv(EXPORTS / 'responses_model_inst_blind.csv')

print(f'pair_cache: {len(pc):,} rows')
print(f'human: {len(human):,} rows')
print(f'model_ib: {len(model_ib):,} rows')
print(f'7/8B models: {len(_7b)}')

pair_cache: 770,033 rows
human: 13,560 rows
model_ib: 12,548 rows
7/8B models: 16


In [2]:
# ── Build per-question diagnostic table ──────────────────────────────────
rows = []
for v in VARIANT_ORDER:
    sub = pc[pc['variant'] == v]
    hh = sub[sub['pair_type'] == 'HH'].groupby('question_id')['sbert_score'].mean()
    hm = sub[(sub['pair_type'] == 'HM') & (sub['subject_2'].isin(_7b))].groupby('question_id')['sbert_score'].mean()
    for qid in hh.index:
        rows.append({'question_id': qid, 'variant': v,
                     'hh_sbert': hh.get(qid, np.nan),
                     'hm_sbert': hm.get(qid, np.nan)})
qv = pd.DataFrame(rows)
meta = human[human['variant'] == 'C'].drop_duplicates('question_id')[
    ['question_id', 'ent', 'op']
].set_index('question_id')
qwide = qv.pivot(index='question_id', columns='variant', values=['hh_sbert', 'hm_sbert'])
qwide.columns = ['_'.join(c) for c in qwide.columns]
df = qwide.join(meta)
df['hm_drop_CA'] = df['hm_sbert_C'] - df['hm_sbert_A']

# ── Entropy ──────────────────────────────────────────────────────────────
def answer_entropy(responses):
    counts = responses['response'].value_counts()
    probs = counts / counts.sum()
    return sp_entropy(probs, base=2)

h_ent = human[human['variant'] == 'C'].groupby('question_id').apply(
    answer_entropy, include_groups=False
).rename('human_entropy')
m_ent = model_ib[(model_ib['variant'] == 'C') & (model_ib['model'].isin(_7b))].groupby('question_id').apply(
    answer_entropy, include_groups=False
).rename('model_entropy')

ent_df = pd.DataFrame({'human_entropy': h_ent, 'model_entropy': m_ent}).join(
    df[['hm_sbert_C', 'hm_drop_CA', 'ent', 'op']]
)
for c in ['human_entropy', 'model_entropy', 'hm_sbert_C', 'hm_drop_CA']:
    ent_df = ent_df[ent_df[c].notna() & np.isfinite(ent_df[c])]
ent_df['entropy_gap'] = ent_df['model_entropy'] - ent_df['human_entropy']

# ── Merge low-support entity types ───────────────────────────────────────
ENT_MERGE = {
    'person': 'person', 'animal': 'animal', 'object': 'object', 'food': 'food',
    'other': 'other', 'product': 'other', 'place': 'other',
    'vehicle': 'other', 'text': 'other',
}
ent_df['ent_group'] = ent_df['ent'].map(ENT_MERGE).fillna('other')

print(f'Questions in entropy analysis: {len(ent_df)}')
print(f'Entity groups: {ent_df["ent_group"].value_counts().to_dict()}')

Questions in entropy analysis: 88
Entity groups: {'object': 23, 'other': 22, 'person': 20, 'animal': 12, 'food': 11}


In [3]:
def export_latex(df, filename, caption, label, float_format='%.3f'):
    """Export a DataFrame to a .tex file with table/tabular wrapping."""
    path = OUT_DIR / filename
    latex = df.to_latex(float_format=float_format, escape=False)
    with open(path, 'w') as f:
        f.write('\\begin{table}[t]\n\\centering\n')
        f.write(f'\\caption{{{caption}}}\n')
        f.write(f'\\label{{{label}}}\n')
        f.write('\\small\n')
        f.write(latex)
        f.write('\\end{table}\n')
    print(f'Exported: {path}')
    # Also print the LaTeX for inline review
    print(latex)

## Table 1: Per-Entity-Group Summary Statistics

In [4]:
ent_order = ['person', 'animal', 'object', 'food', 'other']
t1_rows = []
for g in ent_order:
    sub = ent_df[ent_df['ent_group'] == g]
    t1_rows.append({
        'Entity': g.capitalize(),
        'N': len(sub),
        'H(human)': sub.human_entropy.mean(),
        'H(model)': sub.model_entropy.mean(),
        'Gap': sub.entropy_gap.mean(),
        'HM SBERT': sub.hm_sbert_C.mean(),
        r'C$\to$A deg.': sub.hm_drop_CA.mean(),
    })
# Add ALL row
t1_rows.append({
    'Entity': 'All',
    'N': len(ent_df),
    'H(human)': ent_df.human_entropy.mean(),
    'H(model)': ent_df.model_entropy.mean(),
    'Gap': ent_df.entropy_gap.mean(),
    'HM SBERT': ent_df.hm_sbert_C.mean(),
    r'C$\to$A deg.': ent_df.hm_drop_CA.mean(),
})

t1 = pd.DataFrame(t1_rows).set_index('Entity')
t1['N'] = t1['N'].astype(int)
t1_display = t1.style.format({
    'N': '{:d}', 'H(human)': '{:.3f}', 'H(model)': '{:.3f}',
    'Gap': '{:+.3f}', 'HM SBERT': '{:.3f}', r'C$\to$A deg.': '{:+.3f}',
})
display(t1_display)

export_latex(
    t1, 'table_entropy_entity_summary.tex',
    r'Per-entity-group entropy and alignment statistics (variant~C, 7/8B, $q=88$ free-text). '
    r'Gap $=$ H(model) $-$ H(human); negative values indicate model overconfidence.',
    'tab:entropy_entity'
)

,N,H(human),H(model),Gap,HM SBERT,C$\to$A deg.
Entity,,,,,,
Person,20,3.664,2.486,-1.178,0.421,+0.048
Animal,12,3.475,2.176,-1.298,0.494,+0.095
Object,23,3.260,2.200,-1.060,0.470,+0.070
Food,11,3.364,2.213,-1.151,0.527,+0.137
Other,22,3.755,2.224,-1.531,0.434,+0.065
All,88,3.518,2.269,-1.248,0.460,+0.076


Exported: /home/david/Desktop/yuna/HPA/figures/question_diagnostic/table_entropy_entity_summary.tex
\begin{tabular}{lrrrrrr}
\toprule
 & N & H(human) & H(model) & Gap & HM SBERT & C$\to$A deg. \\
Entity &  &  &  &  &  &  \\
\midrule
Person & 20 & 3.664 & 2.486 & -1.178 & 0.421 & 0.048 \\
Animal & 12 & 3.475 & 2.176 & -1.298 & 0.494 & 0.095 \\
Object & 23 & 3.260 & 2.200 & -1.060 & 0.470 & 0.070 \\
Food & 11 & 3.364 & 2.213 & -1.151 & 0.527 & 0.137 \\
Other & 22 & 3.755 & 2.224 & -1.531 & 0.434 & 0.065 \\
All & 88 & 3.518 & 2.269 & -1.248 & 0.460 & 0.076 \\
\bottomrule
\end{tabular}



## Table 2: Global Correlations

In [5]:
r1, p1 = stats.pearsonr(ent_df['human_entropy'], ent_df['model_entropy'])
r2, p2 = stats.pearsonr(ent_df['human_entropy'], ent_df['hm_sbert_C'])
valid = ent_df[ent_df['entropy_gap'].notna() & ent_df['hm_drop_CA'].notna()]
r3, p3 = stats.pearsonr(valid['entropy_gap'], valid['hm_drop_CA'])

def fmt_p(v):
    if v < 0.001:
        return f'{v:.1e}'
    return f'{v:.3f}'

t2 = pd.DataFrame([
    {'Pair': 'H(human) vs H(model)', 'r': f'{r1:.3f}', 'p': fmt_p(p1), 'N': len(ent_df), 'Interpretation': 'Shared uncertainty'},
    {'Pair': 'H(human) vs HM SBERT', 'r': f'{r2:.3f}', 'p': fmt_p(p2), 'N': len(ent_df), 'Interpretation': 'Consensus drives alignment'},
    {'Pair': r'Entropy gap vs C$\to$A', 'r': f'{r3:.3f}', 'p': fmt_p(p3), 'N': len(valid), 'Interpretation': 'Orthogonal mechanisms'},
]).set_index('Pair')

display(t2)

export_latex(
    t2, 'table_entropy_correlations.tex',
    r'Global correlations between answer entropy and alignment metrics '
    r'(variant~C, 7/8B, $q=88$ free-text). All Pearson $r$.',
    'tab:entropy_corr',
    float_format='%s'
)

,r,p,N,Interpretation
Pair,,,,
H(human) vs H(model),0.554,2.1e-08,88,Shared uncertainty
H(human) vs HM SBERT,-0.848,2.0e-25,88,Consensus drives alignment
Entropy gap vs C$\to$A,0.125,0.246,88,Orthogonal mechanisms


Exported: /home/david/Desktop/yuna/HPA/figures/question_diagnostic/table_entropy_correlations.tex
\begin{tabular}{lllrl}
\toprule
 & r & p & N & Interpretation \\
Pair &  &  &  &  \\
\midrule
H(human) vs H(model) & 0.554 & 2.1e-08 & 88 & Shared uncertainty \\
H(human) vs HM SBERT & -0.848 & 2.0e-25 & 88 & Consensus drives alignment \\
Entropy gap vs C$\to$A & 0.125 & 0.246 & 88 & Orthogonal mechanisms \\
\bottomrule
\end{tabular}



## Table 3: Paired t-test — Human vs Model Entropy (by entity group)

In [6]:
t3_rows = []
for g in ent_order + ['All']:
    sub = ent_df if g == 'All' else ent_df[ent_df['ent_group'] == g]
    t_stat, p_val = stats.ttest_rel(sub['human_entropy'], sub['model_entropy'])
    n_higher = (sub['entropy_gap'] < 0).sum()
    t3_rows.append({
        'Entity': g.capitalize() if g != 'All' else 'All',
        'N': len(sub),
        't': t_stat,
        'p': p_val,
        'Mean gap': sub.entropy_gap.mean(),
        'H $>$ M': f'{n_higher}/{len(sub)}',
    })

t3 = pd.DataFrame(t3_rows).set_index('Entity')
t3_display = t3.style.format({
    'N': '{:d}', 't': '{:.2f}', 'p': fmt_p, 'Mean gap': '{:+.3f}',
})
display(t3_display)

# Pre-format for LaTeX
t3_tex = t3.copy()
t3_tex['p'] = t3_tex['p'].apply(fmt_p)
t3_tex['t'] = t3_tex['t'].apply(lambda x: f'{x:.2f}')
t3_tex['Mean gap'] = t3_tex['Mean gap'].apply(lambda x: f'{x:+.3f}')
t3_tex['N'] = t3_tex['N'].astype(int)
export_latex(
    t3_tex, 'table_entropy_ttest.tex',
    r'Paired $t$-test: human answer entropy vs.\ model answer entropy by entity group '
    r'(variant~C, 7/8B). H~$>$~M = questions where human entropy exceeds model entropy.',
    'tab:entropy_ttest',
    float_format='%s'
)

,N,t,p,Mean gap,H $>$ M
Entity,,,,,
Person,20,5.32,3.9e-05,-1.178,19/20
Animal,12,6.79,3.0e-05,-1.298,12/12
Object,23,6.11,3.7e-06,-1.060,21/23
Food,11,6.24,9.7e-05,-1.151,11/11
Other,22,10.25,1.3e-09,-1.531,22/22
All,88,14.66,3.1e-25,-1.248,85/88


Exported: /home/david/Desktop/yuna/HPA/figures/question_diagnostic/table_entropy_ttest.tex
\begin{tabular}{lrllll}
\toprule
 & N & t & p & Mean gap & H $>$ M \\
Entity &  &  &  &  &  \\
\midrule
Person & 20 & 5.32 & 3.9e-05 & -1.178 & 19/20 \\
Animal & 12 & 6.79 & 3.0e-05 & -1.298 & 12/12 \\
Object & 23 & 6.11 & 3.7e-06 & -1.060 & 21/23 \\
Food & 11 & 6.24 & 9.7e-05 & -1.151 & 11/11 \\
Other & 22 & 10.25 & 1.3e-09 & -1.531 & 22/22 \\
All & 88 & 14.66 & 3.1e-25 & -1.248 & 85/88 \\
\bottomrule
\end{tabular}



## Table 4: Per-Entity Correlation — Human Entropy vs HM SBERT

In [7]:
t4_rows = []
for g in ent_order:
    sub = ent_df[ent_df['ent_group'] == g]
    if len(sub) > 3:
        r, p = stats.pearsonr(sub['human_entropy'], sub['hm_sbert_C'])
        t4_rows.append({'Entity': g.capitalize(), 'N': len(sub),
                        'r': f'{r:+.3f}', 'p': fmt_p(p)})

t4 = pd.DataFrame(t4_rows).set_index('Entity')
display(t4)

export_latex(
    t4, 'table_entropy_sbert_by_entity.tex',
    r'Per-entity-group Pearson $r$: human answer entropy vs.\ HM SBERT '
    r'(variant~C, 7/8B). The consensus$\to$alignment relationship holds within every group.',
    'tab:entropy_sbert_entity',
    float_format='%s'
)

,N,r,p
Entity,,,
Person,20,-0.840,3.6e-06
Animal,12,-0.885,1.3e-04
Object,23,-0.884,2.3e-08
Food,11,-0.802,0.003
Other,22,-0.851,5.3e-07


Exported: /home/david/Desktop/yuna/HPA/figures/question_diagnostic/table_entropy_sbert_by_entity.tex
\begin{tabular}{lrll}
\toprule
 & N & r & p \\
Entity &  &  &  \\
\midrule
Person & 20 & -0.840 & 3.6e-06 \\
Animal & 12 & -0.885 & 1.3e-04 \\
Object & 23 & -0.884 & 2.3e-08 \\
Food & 11 & -0.802 & 0.003 \\
Other & 22 & -0.851 & 5.3e-07 \\
\bottomrule
\end{tabular}



## Table 5: Per-Entity Correlation — Entropy Gap vs C→A Degradation

In [8]:
t5_rows = []
for g in ent_order:
    sub = ent_df[ent_df['ent_group'] == g]
    sub = sub[sub['entropy_gap'].notna() & sub['hm_drop_CA'].notna()]
    if len(sub) > 3:
        r, p = stats.pearsonr(sub['entropy_gap'], sub['hm_drop_CA'])
        t5_rows.append({'Entity': g.capitalize(), 'N': len(sub),
                        'r': f'{r:+.3f}', 'p': fmt_p(p)})

t5 = pd.DataFrame(t5_rows).set_index('Entity')
display(t5)

export_latex(
    t5, 'table_entropy_gap_vs_degradation.tex',
    r'Per-entity-group Pearson $r$: entropy gap (model $-$ human) vs.\ C$\to$A '
    r'degradation (variant~C, 7/8B). No group shows a significant correlation, '
    r'confirming that overconfidence and anchor dependence are orthogonal.',
    'tab:entropy_gap_entity',
    float_format='%s'
)

,N,r,p
Entity,,,
Person,20,+0.127,0.594
Animal,12,+0.276,0.386
Object,23,+0.278,0.198
Food,11,+0.491,0.125
Other,22,-0.247,0.267


Exported: /home/david/Desktop/yuna/HPA/figures/question_diagnostic/table_entropy_gap_vs_degradation.tex
\begin{tabular}{lrll}
\toprule
 & N & r & p \\
Entity &  &  &  \\
\midrule
Person & 20 & +0.127 & 0.594 \\
Animal & 12 & +0.276 & 0.386 \\
Object & 23 & +0.278 & 0.198 \\
Food & 11 & +0.491 & 0.125 \\
Other & 22 & -0.247 & 0.267 \\
\bottomrule
\end{tabular}



## Table 6: Per-Operation-Type Summary Statistics

In [9]:
t6_rows = []
for op in sorted(ent_df['op'].unique()):
    sub = ent_df[ent_df['op'] == op]
    t6_rows.append({
        'Operation': op,
        'N': len(sub),
        'H(human)': sub.human_entropy.mean(),
        'H(model)': sub.model_entropy.mean(),
        'Gap': sub.entropy_gap.mean(),
        'HM SBERT': sub.hm_sbert_C.mean(),
        r'C$\to$A deg.': sub.hm_drop_CA.mean(),
    })
# Add ALL row
t6_rows.append({
    'Operation': 'All',
    'N': len(ent_df),
    'H(human)': ent_df.human_entropy.mean(),
    'H(model)': ent_df.model_entropy.mean(),
    'Gap': ent_df.entropy_gap.mean(),
    'HM SBERT': ent_df.hm_sbert_C.mean(),
    r'C$\to$A deg.': ent_df.hm_drop_CA.mean(),
})

t6 = pd.DataFrame(t6_rows).set_index('Operation')
t6['N'] = t6['N'].astype(int)
t6_display = t6.style.format({
    'N': '{:d}', 'H(human)': '{:.3f}', 'H(model)': '{:.3f}',
    'Gap': '{:+.3f}', 'HM SBERT': '{:.3f}', r'C$\to$A deg.': '{:+.3f}',
})
display(t6_display)

export_latex(
    t6, 'table_entropy_op_summary.tex',
    r'Per-operation-type entropy and alignment statistics (variant~C, 7/8B, $q=88$ free-text). '
    r'Identity questions show the highest C$\to$A degradation (+0.216), '
    r'confirming entity-noun dependence.',
    'tab:entropy_op'
)

,N,H(human),H(model),Gap,HM SBERT,C$\to$A deg.
Operation,,,,,,
act,3,3.039,1.646,-1.394,0.510,+0.027
attr,29,3.579,2.304,-1.275,0.444,+0.061
cause,1,4.622,2.906,-1.716,0.276,-0.033
comp,2,4.441,3.174,-1.267,0.335,+0.012
count,22,2.979,1.835,-1.143,0.539,+0.062
ident,9,3.641,2.502,-1.139,0.495,+0.216
know,2,2.807,2.203,-0.605,0.532,+0.145
spat,12,3.801,2.449,-1.353,0.391,+0.100
temp,3,4.106,2.944,-1.162,0.422,-0.006


Exported: /home/david/Desktop/yuna/HPA/figures/question_diagnostic/table_entropy_op_summary.tex
\begin{tabular}{lrrrrrr}
\toprule
 & N & H(human) & H(model) & Gap & HM SBERT & C$\to$A deg. \\
Operation &  &  &  &  &  &  \\
\midrule
act & 3 & 3.039 & 1.646 & -1.394 & 0.510 & 0.027 \\
attr & 29 & 3.579 & 2.304 & -1.275 & 0.444 & 0.061 \\
cause & 1 & 4.622 & 2.906 & -1.716 & 0.276 & -0.033 \\
comp & 2 & 4.441 & 3.174 & -1.267 & 0.335 & 0.012 \\
count & 22 & 2.979 & 1.835 & -1.143 & 0.539 & 0.062 \\
ident & 9 & 3.641 & 2.502 & -1.139 & 0.495 & 0.216 \\
know & 2 & 2.807 & 2.203 & -0.605 & 0.532 & 0.145 \\
spat & 12 & 3.801 & 2.449 & -1.353 & 0.391 & 0.100 \\
temp & 3 & 4.106 & 2.944 & -1.162 & 0.422 & -0.006 \\
text & 5 & 4.262 & 2.638 & -1.624 & 0.364 & 0.004 \\
All & 88 & 3.518 & 2.269 & -1.248 & 0.460 & 0.076 \\
\bottomrule
\end{tabular}



## Summary

Key takeaways:
1. **Systematic overconfidence**: Models have lower answer entropy than humans on 85/88 questions (paired t = 14.66, p = 3.1e-25)
2. **Consensus drives alignment**: r = -0.85 between human entropy and HM SBERT, holding within every entity group (r = -0.80 to -0.89)
3. **Orthogonal mechanisms**: Entropy gap vs C→A degradation is null globally (r = 0.12) and within every entity group (all p > 0.13)
4. **Identity questions** are the most anchor-dependent (C→A = +0.216), **food** questions have the highest alignment (SBERT = 0.527)

LaTeX tables exported to `figures/question_diagnostic/table_entropy_*.tex`

## Bootstrapped Entropy Comparison (Sample-Size-Matched)

The raw entropy comparison (Tables 1, 3) is confounded by sample size: 40 humans vs ~16 models.
To make a fair comparison, we bootstrap human subsamples matched to each model group's size
and compute per-question entropy with 95% CIs.

## Qualitative Examples by Operation and Entity Type

Top questions by C→A degradation within each category, showing where
entity-anchor dependence is strongest.

In [10]:
# ── Build qualitative examples table ─────────────────────────────────────
meta_full = human[human['variant'] == 'C'].drop_duplicates('question_id')[
    ['question_id', 'question_en', 'ent', 'op', 'gt']
].set_index('question_id')
# gt is a string column with pipe-separated answers
meta_full['gt_short'] = meta_full['gt'].apply(
    lambda x: str(x).split(' | ')[0] if pd.notna(x) else '')

hm_vC = pc[(pc['variant'] == 'C') & (pc['pair_type'] == 'HM') & 
            (pc['subject_2'].isin(_7b))].groupby('question_id')['sbert_score'].mean()
hm_vA = pc[(pc['variant'] == 'A') & (pc['pair_type'] == 'HM') & 
            (pc['subject_2'].isin(_7b))].groupby('question_id')['sbert_score'].mean()
hh_vC = pc[(pc['variant'] == 'C') & (pc['pair_type'] == 'HH')].groupby('question_id')['sbert_score'].mean()

DEG_COL = 'C_to_A'
qdf = meta_full.join(hm_vC.rename('HM')).join(hm_vA.rename('HM_A')).join(hh_vC.rename('HH'))
qdf = qdf[qdf['HM'].notna()].copy()
qdf[DEG_COL] = qdf['HM'] - qdf['HM_A']

# ── By operation type ────────────────────────────────────────────────────
print("=== Top 3 by C->A degradation per operation type ===\n")
op_examples = []
for op in ['ident', 'count', 'attr', 'act', 'spat']:
    sub = qdf[qdf['op'] == op].sort_values(DEG_COL, ascending=False)
    n = len(sub)
    mean_hm = sub['HM'].mean()
    mean_deg = sub[DEG_COL].mean()
    print("--- {} (N={}, mean HM={:.3f}, mean C->A={:+.3f}) ---".format(op, n, mean_hm, mean_deg))
    for qid, row in sub.head(3).iterrows():
        q = str(row.question_en)[:55]
        deg_val = row[DEG_COL]
        print('  "{}" | ent={} | HM={:.3f} | HH={:.3f} | C->A={:+.3f}'.format(
            q, row.ent, row.HM, row.HH, deg_val))
        op_examples.append({
            'Question': str(row.question_en)[:50],
            'Op': op, 'Ent': row.ent,
            'HM': row.HM, 'HH': row.HH,
            r'C$\to$A': deg_val,
        })
    print()

op_ex_df = pd.DataFrame(op_examples).set_index('Question')
display(op_ex_df.style.format({
    'HM': '{:.3f}', 'HH': '{:.3f}', r'C$\to$A': '{:+.3f}',
}))

export_latex(
    op_ex_df, 'table_qualitative_by_op.tex',
    r'Top questions by C$\to$A degradation per operation type '
    r'(variant~C, 7/8B). Identity questions show the steepest drops, '
    r'confirming that entity nouns carry the strongest prior signal.',
    'tab:qual_op'
)

=== Top 3 by C->A degradation per operation type ===

--- ident (N=9, mean HM=0.495, mean C->A=+0.216) ---
  "What type of shoe is that?" | ent=product | HM=0.613 | HH=0.653 | C->A=+0.382
  "What type of bear is this?" | ent=animal | HM=0.572 | HH=0.674 | C->A=+0.343
  "What is this player's position called?" | ent=person | HM=0.561 | HH=0.574 | C->A=+0.304

--- count (N=22, mean HM=0.539, mean C->A=+0.062) ---
  "How many people touching the elephant trunk?" | ent=person | HM=0.673 | HH=0.686 | C->A=+0.182
  "How many hot dogs are on this bun?" | ent=food | HM=0.644 | HH=0.770 | C->A=+0.157
  "How many ski poles is this person holding?" | ent=object | HM=0.754 | HH=0.750 | C->A=+0.157

--- attr (N=29, mean HM=0.444, mean C->A=+0.061) ---
  "What type of computer is the cat using?" | ent=product | HM=0.517 | HH=0.462 | C->A=+0.316
  "What is on the man's ear?" | ent=person | HM=0.533 | HH=0.620 | C->A=+0.310
  "What kind of trees are in the picture?" | ent=object | HM=0.458 | HH=0.642 

,Op,Ent,HM,HH,C$\to$A
Question,,,,,
What type of shoe is that?,ident,product,0.613,0.653,+0.382
What type of bear is this?,ident,animal,0.572,0.674,+0.343
What is this player's position called?,ident,person,0.561,0.574,+0.304
How many people touching the elephant trunk?,count,person,0.673,0.686,+0.182
How many hot dogs are on this bun?,count,food,0.644,0.770,+0.157
How many ski poles is this person holding?,count,object,0.754,0.750,+0.157
What type of computer is the cat using?,attr,product,0.517,0.462,+0.316
What is on the man's ear?,attr,person,0.533,0.620,+0.310
What kind of trees are in the picture?,attr,object,0.458,0.642,+0.181


Exported: /home/david/Desktop/yuna/HPA/figures/question_diagnostic/table_qualitative_by_op.tex
\begin{tabular}{lllrrr}
\toprule
 & Op & Ent & HM & HH & C$\to$A \\
Question &  &  &  &  &  \\
\midrule
What type of shoe is that? & ident & product & 0.613 & 0.653 & 0.382 \\
What type of bear is this? & ident & animal & 0.572 & 0.674 & 0.343 \\
What is this player's position called? & ident & person & 0.561 & 0.574 & 0.304 \\
How many people touching the elephant trunk? & count & person & 0.673 & 0.686 & 0.182 \\
How many hot dogs are on this bun? & count & food & 0.644 & 0.770 & 0.157 \\
How many ski poles is this person holding? & count & object & 0.754 & 0.750 & 0.157 \\
What type of computer is the cat using? & attr & product & 0.517 & 0.462 & 0.316 \\
What is on the man's ear? & attr & person & 0.533 & 0.620 & 0.310 \\
What kind of trees are in the picture? & attr & object & 0.458 & 0.642 & 0.181 \\
What is the elephant eating? & act & animal & 0.442 & 0.507 & 0.105 \\
What is this man

# ── By entity type ───────────────────────────────────────────────────────
print("=== Top 3 by C->A degradation per entity type ===\n")
ent_examples = []
for ent in ['person', 'animal', 'object', 'food']:
    sub = qdf[qdf['ent'] == ent].sort_values(DEG_COL, ascending=False)
    n = len(sub)
    mean_hm = sub['HM'].mean()
    mean_deg = sub[DEG_COL].mean()
    print("--- {} (N={}, mean HM={:.3f}, mean C->A={:+.3f}) ---".format(ent, n, mean_hm, mean_deg))
    for qid, row in sub.head(3).iterrows():
        q = str(row.question_en)[:55]
        deg_val = row[DEG_COL]
        print('  "{}" | op={} | HM={:.3f} | HH={:.3f} | C->A={:+.3f}'.format(
            q, row.op, row.HM, row.HH, deg_val))
        ent_examples.append({
            'Question': str(row.question_en)[:50],
            'Ent': ent, 'Op': row.op,
            'HM': row.HM, 'HH': row.HH,
            r'C$\to$A': deg_val,
        })
    print()

ent_ex_df = pd.DataFrame(ent_examples).set_index('Question')
display(ent_ex_df.style.format({
    'HM': '{:.3f}', 'HH': '{:.3f}', r'C$\to$A': '{:+.3f}',
}))

export_latex(
    ent_ex_df, 'table_qualitative_by_entity.tex',
    r'Top questions by C$\to$A degradation per entity type '
    r'(variant~C, 7/8B). Food questions show the highest mean degradation, '
    r'driven by identity sub-type questions where the entity noun is the answer.',
    'tab:qual_entity'
)